# Pipeline de Datos Detallado: Estrategia de Integración Multifuente
**Equipo:** linuxitOS

Este documento presenta la **auditoría técnica integral** de nuestro pipeline de datos (ETL). El objetivo primordial de esta libreta es demostrar la transparencia metodológica, la solidez estadística y la eficiencia computacional de nuestra arquitectura.

### Objetivos Técnicos:
1.  **Trazabilidad:** Seguimiento del dato desde la fuente oficial (SIODS, Microdatos INEGI/CONEVAL) hasta el dataset maestro.
2.  **Integridad Espacial:** Normalización de claves geográficas (`CVEGEO`) a 5 dígitos para garantizar cruces precisos a nivel municipal.
3.  **Optimización:** Implementación de algoritmos de simplificación geométrica para un rendimiento web de alto desempeño.

    

## Módulo 1: Minería de Metadatos SIODS
**Objetivo:** Extracción de indicadores oficiales de la Agenda 2030 (ODS 6).

Este script consulta la API de la plataforma SIODS. Se enfoca en identificar el "Punto Ciego Estatal" de los indicadores **6.1.1 (Agua Potable Segura)**, permitiendo contrastar las metas globales de la ONU con la realidad territorial en México.

In [1]:
import requests
import json
import pandas as pd
import os
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Configuración de la API oculta de SIODS
URL_API = "https://agenda2030.mx/ods/api/Valores/PorClave"
HEADERS = {"Content-Type": "application/json"}

# Indicadores a extraer (actualizado con el descubrimiento del "Punto Ciego Estatal")
# 6.1.1.c = PCveInd: 618, PCveSer: 2073 (Proporción estatal de agua potable segura)
# 6.2.1.c = PCveInd: 620, PCveSer: 2079 (Proporción estatal de saneamiento seguro - asumido como ejemplo cercano, o nos quedamos con el nacional 192 si no tenemos la serie exacta, pero priorizaremos el 618)
INDICADORES_SIODS = [
    {"nombre": "ODS_6.1.1.a_Nacional_Agua_Segura", "PCveInd": 54, "PCveSer": 2015},
    {"nombre": "ODS_6.1.1.c_Estatal_Agua_Segura", "PCveInd": 618, "PCveSer": 2073},
]


def extraer_siods(ind_config):
    payload = {
        "PCveInd": ind_config["PCveInd"],
        "PCveSer": ind_config["PCveSer"],
        "POrden": "DESC",
        "PCveAgrupaCla": "99",
        "PIdioma": "ES",
    }

    try:
        response = requests.post(URL_API, json=payload, headers=HEADERS, verify=False)
        if response.status_code == 200:
            data = response.json()

            series = data.get("Series", [])
            if not series:
                return None

            serie = series[0]
            coberturas = serie.get("Coberturas", [])

            resultados = []
            for cob in coberturas:
                clave_geo = cob.get("ClaveCobGeo_cg", "00")
                desc_geo = cob.get("Descrip_cg", "Nacional")

                for obs in cob.get("ValorDato", []):
                    resultados.append(
                        {
                            "Indicador": ind_config["nombre"],
                            "Clave_Entidad": clave_geo,
                            "Entidad": desc_geo,
                            "Año": obs.get("AADato_ser"),
                            "Valor": obs.get("Dato_ser"),
                        }
                    )

            return pd.DataFrame(resultados)
        else:
            print(f"[ERROR] Código de estado {response.status_code} al consultar indicador {ind_config['nombre']}")
            return None
    except Exception as e:
        print(f"[ERROR] Excepción de conexión: {e}")
        return None


def ejecutar_pipeline():
    print("[INFO] Iniciando extracción de datos desde API SIODS (Agenda 2030).")

    df_list = []
    for config in INDICADORES_SIODS:
        print(f"[INFO] Procesando indicador: {config['nombre']}")
        df = extraer_siods(config)
        if df is not None and not df.empty:
            df_list.append(df)

    if df_list:
        df_final = pd.concat(df_list, ignore_index=True)

        ruta_salida = "../datos/siods_agua_saneamiento_nacional.csv"
        os.makedirs(os.path.dirname(ruta_salida), exist_ok=True)
        df_final.to_csv(ruta_salida, index=False)

        print(f"\n[ÉXITO] Extracción finalizada. Dataset original guardado en: {ruta_salida}")
        print("\n[INFO] --- Resumen Estadístico y Granularidad ---")
        entidades_unicas = df_final["Entidad"].unique()
        print(f"[INFO] Entidades geográficas únicas registradas: {len(entidades_unicas)}")
        print(
            "[INFO] Detalle: La matriz contiene inferencia a nivel Nacional y Estatal. "
            "Requisito pivotar hacia CONEVAL para resolver déficit de datos a nivel municipal."
        )


if __name__ == "__main__":
    # Ajuste de directorio de trabajo para soportar ejecución en Terminal y Notebooks
    if "__file__" in globals():
        # Estamos ejecutando como script (.py)
        # Si el script se ejecuta desde la carpeta scripts/, subimos un nivel a la raíz
        script_dir = os.path.dirname(os.path.abspath(__file__))
        if os.path.basename(script_dir) == 'scripts':
            os.chdir(os.path.join(script_dir, ".."))
    else:
        # Estamos en un entorno interactivo (Jupyter)
        if not os.path.exists('scripts') and os.path.exists(os.path.join('..', 'scripts')):
            os.chdir('..')

    ejecutar_pipeline()


[INFO] Iniciando extracción de datos desde API SIODS (Agenda 2030).
[INFO] Procesando indicador: ODS_6.1.1.a_Nacional_Agua_Segura
[INFO] Procesando indicador: ODS_6.1.1.c_Estatal_Agua_Segura

[ÉXITO] Extracción finalizada. Dataset original guardado en: ../datos/siods_agua_saneamiento_nacional.csv

[INFO] --- Resumen Estadístico y Granularidad ---
[INFO] Entidades geográficas únicas registradas: 33
[INFO] Detalle: La matriz contiene inferencia a nivel Nacional y Estatal. Requisito pivotar hacia CONEVAL para resolver déficit de datos a nivel municipal.


## Módulo 2: Microdatos Censales (INEGI 2020)
**Objetivo:** Definición de la vocación territorial (Rural vs Urbano).

Implementamos un análisis de granularidad máxima utilizando el dataset **ITER 2020**. El algoritmo clasifica cada una de las miles de localidades del país bajo el estándar de 2,500 habitantes de INEGI, agregándolas a nivel municipal para derivar una métrica de "Ruralidad Predominante" (>50% de población en áreas rurales).

In [2]:
import pandas as pd
import numpy as np
import os

# Ajuste de directorio de trabajo para soportar ejecución en Terminal y Notebooks
if not os.path.exists('scripts') and os.path.exists(os.path.join('..', 'scripts')):
    os.chdir('..')

file_path = "datos/conjunto_de_datos_iter_00CSV20.csv.gz"

print("[INFO] Cargando microdatos censales (ITER 2020)...")
df = pd.read_csv(file_path, usecols=['ENTIDAD', 'MUN', 'LOC', 'NOM_LOC', 'POBTOT'], dtype=str)

# Limpieza básica
# Filtramos para quitar el total nacional y estatales (MUN == '000')
df = df[df['MUN'] != '000']
# Filtramos para quitar los totales municipales (LOC == '0000') y quedarnos solo con localidades
df_loc = df[df['LOC'] != '0000'].copy()

# Convertir población a numérico ('*' se vuelve NaN, luego 0)
df_loc['POBTOT'] = pd.to_numeric(df_loc['POBTOT'], errors='coerce').fillna(0)

# Criterio INEGI: Localidad rural = menos de 2500 habitantes
df_loc['es_rural'] = df_loc['POBTOT'] < 2500
df_loc['pob_rural'] = np.where(df_loc['es_rural'], df_loc['POBTOT'], 0)

# Agrupación a nivel municipal
df_loc['CVEGEO'] = df_loc['ENTIDAD'] + df_loc['MUN']
agrupado = df_loc.groupby('CVEGEO').agg(
    poblacion_total=('POBTOT', 'sum'),
    poblacion_rural=('pob_rural', 'sum'),
    total_localidades=('LOC', 'count'),
    localidades_rurales=('es_rural', 'sum')
).reset_index()

# Calcular porcentaje rural del municipio
agrupado['pct_rural'] = (agrupado['poblacion_rural'] / agrupado['poblacion_total']) * 100
agrupado['pct_rural'] = agrupado['pct_rural'].fillna(0)

# Clasificar municipio (Rural si > 50% de su población vive en localidades rurales)
agrupado['clasificacion'] = np.where(agrupado['pct_rural'] > 50, 'Rural', 'Urbano')

print("\n[INFO] --- Resumen Poblacional (Censo 2020) ---")
print(f"[INFO] Municipios totales evaluados: {len(agrupado)}")
print(f"[INFO] Entidades de vocación rural predominante: {len(agrupado[agrupado['clasificacion'] == 'Rural'])}")
print(f"[INFO] Entidades de vocación urbana predominante: {len(agrupado[agrupado['clasificacion'] == 'Urbano'])}")
print(f"[INFO] Universo poblacional censado: {agrupado['poblacion_total'].sum():,.0f}")
print(f"[INFO] Población clasificada como rural: {agrupado['poblacion_rural'].sum():,.0f}")
print(f"[INFO] Tasa global de ruralidad: {(agrupado['poblacion_rural'].sum() / agrupado['poblacion_total'].sum()) * 100:.2f}%")



[INFO] Cargando microdatos censales (ITER 2020)...

[INFO] --- Resumen Poblacional (Censo 2020) ---
[INFO] Municipios totales evaluados: 2469
[INFO] Entidades de vocación rural predominante: 1360
[INFO] Entidades de vocación urbana predominante: 1109
[INFO] Universo poblacional censado: 126,411,503
[INFO] Población clasificada como rural: 27,338,731
[INFO] Tasa global de ruralidad: 21.63%


## Módulo 3: Pobreza Multidimensional (CONEVAL)
**Objetivo:** Extracción de vectores de vulnerabilidad social.

Procesamos la matriz del CONEVAL enfocándonos en la **Pobreza Extrema** y la **Carencia por acceso a los servicios básicos en la vivienda**. Este paso es crítico para establecer la correlación estadística entre la falta de infraestructura hídrica y la marginación estructural.

In [3]:
import pandas as pd

def process_coneval():
    import os
    if not os.path.exists('scripts') and os.path.exists(os.path.join('..', 'scripts')):
        os.chdir('..')

    file_path = "datos/Concentrado_indicadores_de_pobreza_2020.xlsx"
    print(f"[INFO] Leyendo matriz CONEVAL: {file_path}")
    
    df = pd.read_excel(file_path, sheet_name="Concentrado municipal", header=None, skiprows=8)
    
    # 3: CVEGEO
    # 4: Municipio
    # 10: Pobreza 2020 (%)
    # 19: Pobreza extrema 2020 (%) -> NUEVA COLUMNA AGREGADA PARA MAYOR IMPACTO
    # 94: Carencia servicios basicos 2020 (%)
    df_clean = df[[3, 4, 10, 19, 94]].copy()
    df_clean.columns = ["CVEGEO", "Municipio", "Pobreza_pct", "Pobreza_extrema_pct", "Carencia_servicios_pct"]
    
    df_clean.dropna(subset=["CVEGEO"], inplace=True)
    df_clean["CVEGEO"] = df_clean["CVEGEO"].astype(str).str.replace(".0", "", regex=False).str.zfill(5)
    df_clean = df_clean[df_clean["CVEGEO"].str.len() == 5]
    
    df_clean["Pobreza_pct"] = pd.to_numeric(df_clean["Pobreza_pct"], errors='coerce')
    df_clean["Pobreza_extrema_pct"] = pd.to_numeric(df_clean["Pobreza_extrema_pct"], errors='coerce')
    df_clean["Carencia_servicios_pct"] = pd.to_numeric(df_clean["Carencia_servicios_pct"], errors='coerce')
    
    out_path = "datos/coneval_clean_2020.csv"
    df_clean.to_csv(out_path, index=False)
    print(f"[ÉXITO] Matriz CONEVAL sanitizada y almacenada en: {out_path}")
    
    # Estadísticas rápidas para el análisis
    print("\n[INFO] --- Resumen Descriptivo (CONEVAL) ---")
    print(df_clean.describe())

if __name__ == "__main__":
    process_coneval()


[INFO] Leyendo matriz CONEVAL: datos/Concentrado_indicadores_de_pobreza_2020.xlsx
[ÉXITO] Matriz CONEVAL sanitizada y almacenada en: datos/coneval_clean_2020.csv

[INFO] --- Resumen Descriptivo (CONEVAL) ---
       Pobreza_pct  Pobreza_extrema_pct  Carencia_servicios_pct
count  2466.000000          2466.000000             2466.000000
mean     62.002065            17.236355               40.134817
std      21.903723            15.323598               29.865334
min       5.450951             0.000000                0.084550
25%      45.580691             5.354647               11.440258
50%      62.745101            12.537985               35.090204
75%      80.316135            24.275407               66.028980
max      99.646676            84.447850              100.000000


## Módulo 4: Integración del Dataset Maestro (Baseline)
**Objetivo:** Fusión de las dimensiones sociodemográficas y de pobreza.

Realizamos un *Left Join* determinista utilizando la clave `CVEGEO` como eje de unión. Este módulo garantiza la **Integridad Referencial** de nuestro proyecto, asegurando que cada municipio rural identificado tenga su correspondiente vector de pobreza alineado temporalmente al ciclo 2020.

In [4]:
import pandas as pd
import numpy as np
import os

def procesar_inegi():
    print("[INFO] 1. Agregando y calculando matriz de ruralidad municipal (Censo INEGI 2020)")
    file_path = "datos/conjunto_de_datos_iter_00CSV20.csv.gz"
    
    # Leemos solo las columnas necesarias para ahorrar memoria
    df = pd.read_csv(file_path, usecols=['ENTIDAD', 'NOM_ENT', 'MUN', 'LOC', 'POBTOT'], dtype=str)
    
    # Filtros: quitar totales estatales (MUN='000') y totales municipales (LOC='0000')
    df = df[df['MUN'] != '000']
    df_loc = df[df['LOC'] != '0000'].copy()
    
    # Convertir población a numérico (reemplazando '*' por NaN y luego 0)
    df_loc['POBTOT'] = pd.to_numeric(df_loc['POBTOT'], errors='coerce').fillna(0)
    
    # Regla de ruralidad INEGI: Localidad con menos de 2500 habitantes
    df_loc['es_rural'] = df_loc['POBTOT'] < 2500
    df_loc['pob_rural'] = np.where(df_loc['es_rural'], df_loc['POBTOT'], 0)
    
    # Agrupar por clave municipal (CVEGEO = ENTIDAD + MUN)
    df_loc['CVEGEO'] = df_loc['ENTIDAD'] + df_loc['MUN']
    
    agrupado = df_loc.groupby('CVEGEO').agg(
        Estado=('NOM_ENT', 'first'),
        poblacion_total=('POBTOT', 'sum'),
        poblacion_rural=('pob_rural', 'sum')
    ).reset_index()
    
    # Calcular porcentaje rural del municipio
    agrupado['pct_rural'] = (agrupado['poblacion_rural'] / agrupado['poblacion_total']) * 100
    agrupado['pct_rural'] = agrupado['pct_rural'].fillna(0)
    
    # Clasificación principal para el storytelling
    agrupado['clasificacion_rural'] = np.where(agrupado['pct_rural'] > 50, 'Rural', 'Urbano')
    
    return agrupado[['CVEGEO', 'Estado', 'poblacion_total', 'pct_rural', 'clasificacion_rural']]

def merge_datos():
    inegi_df = procesar_inegi()
    
    coneval_path = "datos/coneval_clean_2020.csv"
    print(f"[INFO] 2. Incorporando vector de pobreza multidimensional CONEVAL: {coneval_path}")
    coneval_df = pd.read_csv(coneval_path, dtype={'CVEGEO': str})
    
    # Asegurar formato de clave a 5 dígitos
    inegi_df['CVEGEO'] = inegi_df['CVEGEO'].str.zfill(5)
    coneval_df['CVEGEO'] = coneval_df['CVEGEO'].str.zfill(5)
    
    print("[INFO] 3. Inicializando join relacional (Inner Merge) entre bases sociodemográficas...")
    # Inner merge para conservar solo los municipios que existen en ambas fuentes
    merged_df = pd.merge(coneval_df, inegi_df, on='CVEGEO', how='inner')
    
    out_path = "datos/final_merged_data.csv"
    merged_df.to_csv(out_path, index=False)
    
    print(f"\n[ÉXITO] Matriz transaccional generada en: {out_path}")
    print(f"[INFO] Observaciones integradas: {len(merged_df)}")
    
    print("\n[INFO] --- Vista Preliminar (Head) ---")
    print(merged_df.head().to_string())
    
    print("\n[INFO] --- Agrupación Demográfica ---")
    resumen = merged_df.groupby('clasificacion_rural').agg(
        Carencia_Agua_Promedio=('Carencia_servicios_pct', 'mean'),
        Pobreza_Extrema_Promedio=('Pobreza_extrema_pct', 'mean'),
        Municipios=('CVEGEO', 'count')
    ).round(2)
    print(resumen.to_string())

if __name__ == "__main__":
    # Ajuste de directorio de trabajo para soportar ejecución en Terminal y Notebooks
    if "__file__" in globals():
        script_dir = os.path.dirname(os.path.abspath(__file__))
        if os.path.basename(script_dir) == 'scripts':
            os.chdir(os.path.join(script_dir, ".."))
    else:
        if not os.path.exists('scripts') and os.path.exists(os.path.join('..', 'scripts')):
            os.chdir('..')

    merge_datos()


[INFO] 1. Agregando y calculando matriz de ruralidad municipal (Censo INEGI 2020)
[INFO] 2. Incorporando vector de pobreza multidimensional CONEVAL: datos/coneval_clean_2020.csv
[INFO] 3. Inicializando join relacional (Inner Merge) entre bases sociodemográficas...

[ÉXITO] Matriz transaccional generada en: datos/final_merged_data.csv
[INFO] Observaciones integradas: 2469

[INFO] --- Vista Preliminar (Head) ---
  CVEGEO       Municipio  Pobreza_pct  Pobreza_extrema_pct  Carencia_servicios_pct          Estado  poblacion_total  pct_rural clasificacion_rural
0  01001  Aguascalientes    23.682258             1.974081                1.135570  Aguascalientes           951537   5.264220              Urbano
1  01002        Asientos    40.131881             4.142806                7.411217  Aguascalientes            51896  65.777709               Rural
2  01003        Calvillo    45.755944             4.498973                3.078508  Aguascalientes            58486  51.497794               Rura

## Módulo 5: Derechos de Agua e Ingresos Municipales (SHCP)
**Objetivo:** Análisis de la salud fiscal y recaudación hídrica.

Añadimos la dimensión financiera mediante los datos de la SHCP sobre **Derechos de Agua**. Calculamos el monto recaudado per cápita para identificar desigualdades municipales entre la capacidad administrativa y el acceso efectivo al recurso.

In [5]:
import pandas as pd
import os


def main():
    # Definir rutas absolutas basadas en la estructura del proyecto
    if "__file__" in globals():
        base_dir = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
    else:
        base_dir = os.getcwd()
        if not os.path.exists(os.path.join(base_dir, 'scripts')):
            base_dir = os.path.dirname(base_dir)

    merged_path = os.path.join(base_dir, "datos", "final_merged_data.csv")
    shcp_path = os.path.join(base_dir, "datos", "derecho_agua_municipal.csv")

    print("[INFO] Instanciando dataframes (Dataset Maestro y SHCP)...")
    # Leer dataset principal (manteniendo CVEGEO como string para no perder ceros a la izquierda)
    df_main = pd.read_csv(merged_path, dtype={"CVEGEO": str})
    # Leer dataset de SHCP
    df_shcp = pd.read_csv(shcp_path)

    print("[INFO] Limpiando base de derechos de agua SHCP (Corte: 2020)...")
    # Filtrar solo el ciclo 2020 para alinear temporalmente con INEGI y CONEVAL
    df_shcp_2020 = df_shcp[df_shcp["ciclo"] == 2020].copy()

    # Construir la clave CVEGEO: 2 dígitos entidad + 3 dígitos municipio
    df_shcp_2020["CVEGEO"] = df_shcp_2020["id_entidad_federativa"].astype(
        str
    ).str.zfill(2) + df_shcp_2020["id_municipio"].astype(str).str.zfill(3)

    # Seleccionar columnas de interés para no saturar el dataset maestro
    cols_to_keep = [
        "CVEGEO",
        "monto_agua",
        "monto_recaudado_percapita",
        "tomas_pagadas",
    ]
    df_shcp_2020 = df_shcp_2020[cols_to_keep]

    # Limpiar las columnas financieras/tomas: Convertir a numérico y rellenar nulos con 0
    for col in ["monto_agua", "monto_recaudado_percapita", "tomas_pagadas"]:
        df_shcp_2020[col] = pd.to_numeric(df_shcp_2020[col], errors="coerce").fillna(0)

    print("[INFO] Ejecutando operación Left Join con estructura base...")
    # Realizar un left join para mantener todos los municipios que ya teníamos,
    # y simplemente añadir la información de recaudación.
    df_final = pd.merge(df_main, df_shcp_2020, on="CVEGEO", how="left")

    # Rellenar con 0 si algún municipio del main no existía en el padrón de SHCP
    for col in ["monto_agua", "monto_recaudado_percapita", "tomas_pagadas"]:
        df_final[col] = df_final[col].fillna(0)

    # Sobrescribir el dataset final
    print(f"[INFO] Escribiendo matriz transaccional en {merged_path}...")
    df_final.to_csv(merged_path, index=False)

    print(f"[ÉXITO] Dimensiones financieras integradas al espacio matricial.")
    print(f"[INFO] Verificando cabeceras financieras (n=5):")
    print(df_final[["CVEGEO", "Municipio", "monto_agua", "tomas_pagadas"]].head())


if __name__ == "__main__":
    main()


[INFO] Instanciando dataframes (Dataset Maestro y SHCP)...
[INFO] Limpiando base de derechos de agua SHCP (Corte: 2020)...
[INFO] Ejecutando operación Left Join con estructura base...
[INFO] Escribiendo matriz transaccional en /home/sebs/Documentos/HackODS/linuxitOS-HackODS2026/datos/final_merged_data.csv...
[ÉXITO] Dimensiones financieras integradas al espacio matricial.
[INFO] Verificando cabeceras financieras (n=5):
  CVEGEO       Municipio   monto_agua  tomas_pagadas
0  01001  Aguascalientes  893532100.0       242484.0
1  01002        Asientos    2961854.0         2989.0
2  01003        Calvillo   38814809.0        16877.0
3  01004           Cosío    1047418.0          774.0
4  01005     Jesús María  103769889.0        28630.0


## Módulo 6: Disponibilidad Hídrica y Regiones (CONAGUA)
**Objetivo:** Contextualización hidro-topográfica.

Integramos las **Regiones Hidrológico-Administrativas (RHA)** y el indicador de cobertura de la CONAGUA. Derivamos una métrica de "Deficiencia Hídrica Específica" (100 - %cobertura), contrastando la visión de la autoridad hídrica con los datos socioeconómicos previos.

In [6]:
import pandas as pd

def main():
    import os
    if not os.path.exists('scripts') and os.path.exists(os.path.join('..', 'scripts')):
        os.chdir('..')

    print("[INFO] Leyendo base: final_merged_data.csv")
    df_main = pd.read_csv("datos/final_merged_data.csv")
    
    # Asegurar que CVEGEO sea string de 5 caracteres
    df_main['CVEGEO'] = df_main['CVEGEO'].astype(str).str.zfill(5)
    
    print("[INFO] Leyendo base: CONAGUA (Población con acceso al agua 2020)")
    df_conagua = pd.read_excel("datos/Población con acceso al agua en el año 2020.xlsx")
    
    # Crear CVEGEO en CONAGUA
    df_conagua['CVEGEO'] = df_conagua['Clave Entidad'].astype(str).str.zfill(2) + df_conagua['Clave Municipio'].astype(str).str.zfill(3)
    
    # Calcular carencia de agua
    df_conagua['carencia_agua_conagua_pct'] = 100.0 - df_conagua['Población Cobertura(%)']
    df_conagua['cobertura_agua_conagua_pct'] = df_conagua['Población Cobertura(%)']
    
    # Seleccionar solo las columnas necesarias (regla de oro: nada de demografía de CONAGUA)
    cols_to_keep = ['CVEGEO', 'Clave RHA', 'RHA', 'cobertura_agua_conagua_pct', 'carencia_agua_conagua_pct']
    df_conagua_subset = df_conagua[cols_to_keep]
    
    print("[INFO] Concatenando observaciones (Left Join por CVEGEO)...")
    df_merged = pd.merge(df_main, df_conagua_subset, on='CVEGEO', how='left')
    
    print(f"[INFO] Observaciones iniciales: {len(df_main)}")
    print(f"[INFO] Observaciones resultantes: {len(df_merged)}")
    print("[INFO] Verificación taxonómica (head):")
    print(df_merged[['CVEGEO', 'Municipio', 'RHA', 'cobertura_agua_conagua_pct', 'carencia_agua_conagua_pct']].head(3))
    
    print("[INFO] Volcando estructura consolidada en disco (CSV)...")
    df_merged.to_csv("datos/final_merged_data.csv", index=False)
    print("[ÉXITO] Base CONAGUA unificada con dataset maestro.")

if __name__ == "__main__":
    main()


[INFO] Leyendo base: final_merged_data.csv
[INFO] Leyendo base: CONAGUA (Población con acceso al agua 2020)
[INFO] Concatenando observaciones (Left Join por CVEGEO)...
[INFO] Observaciones iniciales: 2469
[INFO] Observaciones resultantes: 2469
[INFO] Verificación taxonómica (head):
  CVEGEO       Municipio                      RHA  cobertura_agua_conagua_pct  \
0  01001  Aguascalientes  Lerma-Santiago-Pacífico                       99.51   
1  01002        Asientos  Lerma-Santiago-Pacífico                       99.09   
2  01003        Calvillo  Lerma-Santiago-Pacífico                       99.13   

   carencia_agua_conagua_pct  
0                       0.49  
1                       0.91  
2                       0.87  
[INFO] Volcando estructura consolidada en disco (CSV)...
[ÉXITO] Base CONAGUA unificada con dataset maestro.


## Módulo 7: Optimización de Activos (Douglas-Peucker)
**Objetivo:** Eficiencia computacional y performance del Dashboard.

Para garantizar un tiempo de carga menor a 2 segundos en el navegador, implementamos el algoritmo de **Douglas-Peucker**. Este módulo simplifica la topología de las cuencas y estados de México, reduciendo el peso de los archivos GeoJSON de **18MB a ~1MB** sin perder la precisión necesaria para la visualización del Dashboard.

In [7]:
import sys
try:
    import geopandas as gpd
except ImportError:
    print("[FATAL] Dependencia requerida no encontrada: Geopandas no está instalado en el entorno.")
    sys.exit(1)

import os

if "__file__" in globals():
    BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
else:
    BASE_DIR = os.getcwd()
    if not os.path.exists(os.path.join(BASE_DIR, 'scripts')):
        BASE_DIR = os.path.dirname(BASE_DIR)

files_to_process = [
    ('Disponibilidad_Cuencas_2020.json', 'Disponibilidad_Cuencas_2020_lite.json'),
    ('Estado.json', 'Estado_lite.json')
]

for in_name, out_name in files_to_process:
    input_path = os.path.join(BASE_DIR, 'datos', in_name)
    output_path = os.path.join(BASE_DIR, 'datos', out_name)

    print(f"[INFO] Cargando topología: {in_name}")
    gdf = gpd.read_file(input_path)

    # Simplificar los polígonos conservando la integridad topológica
    # tolerance=0.015 grados approx 1.5 km
    # tolerance=0.015 grados approx 1.5 km
    print(f"[INFO] Aplicando Douglas-Peucker: reduccion poligonal para {in_name}")
    gdf.geometry = gdf.geometry.simplify(tolerance=0.015, preserve_topology=True)

    print(f"[INFO] Vaciando vector espacial optimizado a io stream: {out_name}")
    gdf.to_file(output_path, driver='GeoJSON')
    file_size = os.path.getsize(output_path) / (1024 * 1024)
    print(f"[ÉXITO] Archivo serializado: {output_path}")
    print(f"[INFO] Peso heurístico en disco: {file_size:.2f} MB\n")


[INFO] Cargando topología: Disponibilidad_Cuencas_2020.json
[INFO] Aplicando Douglas-Peucker: reduccion poligonal para Disponibilidad_Cuencas_2020.json
[INFO] Vaciando vector espacial optimizado a io stream: Disponibilidad_Cuencas_2020_lite.json
[ÉXITO] Archivo serializado: /home/sebs/Documentos/HackODS/linuxitOS-HackODS2026/datos/Disponibilidad_Cuencas_2020_lite.json
[INFO] Peso heurístico en disco: 1.00 MB

[INFO] Cargando topología: Estado.json
[INFO] Aplicando Douglas-Peucker: reduccion poligonal para Estado.json
[INFO] Vaciando vector espacial optimizado a io stream: Estado_lite.json
[ÉXITO] Archivo serializado: /home/sebs/Documentos/HackODS/linuxitOS-HackODS2026/datos/Estado_lite.json
[INFO] Peso heurístico en disco: 0.22 MB



## Conclusión Metodológica
Este pipeline representa una solución escalable para la integración de datos gubernamentales heterogéneos. La automatización de la limpieza, el cruce de variables sociodemográficas con indicadores de disponibilidad hídrica y la optimización final de la geometría aseguran que el proyecto no solo sea una visualización estática, sino una plataforma de análisis técnico de alto nivel.

**Próximo Paso:** Instanciación del Dashboard vía `quarto render dashboard/index.qmd`.